In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px




In [21]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [22]:
# transactions: a list of recent orders, provided by the client.
# mail_file: a list of customers who received the mailer, provided by the client.
# holdout_file: a list of addresses taken from the same population as those in the mail file, 
# but these addresses were not sent this mailing campaign. 
# Used as a control group when measuring marketing performance.

transactions = pd.read_csv('data/data-analyst-take-home/dataset1A.csv')
mail_file = pd.read_csv('data/data-analyst-take-home/dataset1B.csv')
holdout_file = pd.read_csv('data/data-analyst-take-home/dataset1C.csv')
mail_plan = pd.read_csv('data/data-analyst-take-home/mail_plan.csv')

**Section 1**

In [23]:
# Calculate the number of valid addresses 
valid_addresses = num_ints = sum(x.isdigit() for x in transactions['PrimaryAddress'].str[0])
print(f"Number of rows in the transactions file that have a valid address: {valid_addresses}")

Number of rows in the transactions file that have a valid address: 9025


In [24]:
# Clean the datatset and calculate the number of rows 
cleaned_transactions = transactions[transactions['PrimaryAddress'].str[0].str.isdigit()]
cleaned_transactions = cleaned_transactions.dropna(subset=['OrderNumber', 'OrderDate'])
cleaned_transactions = cleaned_transactions.sort_values(by='OrderDate').drop_duplicates(subset=['OrderNumber'], keep='first')
print(f"Number of rows in the cleaned transactions file: {len(cleaned_transactions)}")

Number of rows in the cleaned transactions file: 9003


In [25]:
# Find the state with the most orders and how many orders it had
most_popular_state = cleaned_transactions.groupby('State')['OrderNumber'].count()
print(f"{most_popular_state.idxmax()} is the state with the most orders, with {most_popular_state.max()} orders.")

TN is the state with the most orders, with 204 orders.


In [26]:
# How many orders have a secondary address? 
secondary_address_count = cleaned_transactions['SecondaryAddress'].notna().sum()
print(f"Number of orders with a secondary address: {secondary_address_count}")

Number of orders with a secondary address: 4521


In [27]:
# How many unique customers (based on `CustomerID`) placed an order? 
# What is the average number of orders placed per customer?

unique_customers = cleaned_transactions['CustomerID'].nunique()
average_orders_per_customer = cleaned_transactions.groupby('CustomerID')['OrderNumber'].count().mean()
print(f"Number of unique customers: {unique_customers}")
print(f"Average number of orders placed per customer: {average_orders_per_customer:.2f}")

Number of unique customers: 8997
Average number of orders placed per customer: 1.00


6. The `CustomerID` field has some duplicates. Why do you think this is? Is there a circumstance in which you'd remove duplicates for analysis purposes, and if so, how would you determine which orders to keep?

The 'CustomerID' field has some duplicates because some customers make multiple transactions. The case where I would remove duplicates would be if I were only interested in unique customers or a customer's first order. I would determine which orders to keep based on the task at hand. For example, if I wanted to examine the cost of a customer's first order, I would remove the rest of the duplicates. Also, if I were looking at the customer's newest orders. 

7. What other information do you think would be useful to receive from the client in order to analyze the performance of our marketing campaigns? Is there anything that could help for targeting future mailings?

Other information of interest could include customer demographic data such as income level, gender, average income of their neighborhood, and age. This data could help us understand our target audience better. Also, more data about past campaigns and what worked and what didn't would also be beneficial in future campaigns. 


**Section 2**

In [28]:
# How many "matches" are there between the transactions file and the mail file? 
# Between the transactions file and the holdout file?

matches_mail_file = cleaned_transactions['FullAddress'].isin(mail_file['FullAddress']).sum()
matches_holdout_file = cleaned_transactions['FullAddress'].isin(holdout_file['FullAddress']).sum()
print(f"Number of matches between transactions and mail file: {matches_mail_file}")
print(f"Number of matches between transactions and holdout file: {matches_holdout_file}")

Number of matches between transactions and mail file: 1871
Number of matches between transactions and holdout file: 480


In [29]:
# The mailing had various different formats and audience targets, as marked by the `TestCell` field. 
# How many matches are there for each `TestCell`?

matches_by_testcell = cleaned_transactions.merge(mail_file, on='FullAddress', how='inner')['TestCell'].value_counts()
print("Matches by TestCell in Mail and Transactions:")
print(matches_by_testcell)
matches_by_holdout = cleaned_transactions.merge(holdout_file, on='FullAddress', how='inner')['TestCell'].value_counts() 
print("Matches by TestCell in Holdout:")
print(matches_by_holdout)

Matches by TestCell in Mail and Transactions:
TestCell
TEST1    1392
TEST2     467
TEST3      12
Name: count, dtype: int64
Matches by TestCell in Holdout:
TestCell
H-TEST    480
Name: count, dtype: int64


In [30]:
# For each `TestCell`, 
# what percentage of people in the mail and holdout files placed an order?

full_dataset = cleaned_transactions.merge(mail_file, on='FullAddress', how='left').merge(holdout_file, on='FullAddress', how='left', suffixes=('_mail', '_holdout'))
full_dataset_holdout = full_dataset[full_dataset['TestCell_holdout'].notna()]
full_dataset_mail = full_dataset[full_dataset['TestCell_mail'].notna()]
percentage_mail = (full_dataset_mail.groupby('TestCell_mail')['CustomerID'].nunique() / mail_file.groupby('TestCell')['FullAddress'].nunique() * 100)
percentage_holdout = (full_dataset_holdout.groupby('TestCell_holdout')['CustomerID'].nunique() / holdout_file.groupby('TestCell')['FullAddress'].nunique() * 100)
print("Percentage order placed:")
print(percentage_mail, percentage_holdout)

Percentage order placed:
TestCell_mail
TEST1     27.353115
TEST2     12.218734
TEST3    100.000000
dtype: float64 TestCell_holdout
H-TEST    20.78822
dtype: float64


In [31]:
# The mail plan above lists the different Test Cells, the quantity mailed, 
# and the cost to the client. (A CSV version is provided as well, `mail_plan.csv`) 
# Based on this mail plan, create a report or visualization of the campaign's performance that we could share with the client.

# Metrics/key performance indicators include, but are not limited to:
#  - response rate (number of matched orders divided by quantity)
#  - cost per order (CPO) (cost divided by number of matched orders)
#  - return on ad spend (ROAS) (revenue from matched orders divided by campaign cost)
#  - Breakouts for first time orders, audience target, and so on




In [32]:
# Response Rate by Test Cell

fig = px.bar(percentage_mail, x=percentage_mail.index, y=percentage_mail.values, opacity=0.6, labels={'x': 'Test Cell', 'y': 'Response Rate (%)'}, title='Response Rate by Test Cell')
fig.add_hline(
    y=percentage_holdout.values[0],
    line_dash="dash",
    annotation_text="Holdout (Baseline)"
)
fig.show()
                                                                                                                     

In [33]:
summary = full_dataset_mail.groupby("TestCell_mail").agg(
    responders=("CustomerID", "nunique"),
    orders=("OrderNumber", "nunique"),    
    revenue=("OrderRevenue", "sum")
)
plan = mail_plan.copy()
plan["TestCell"] = plan["TestCell"].astype(str).str.strip()
summary.index = summary.index.astype(str).str.strip()

summary = summary.join(plan.set_index("TestCell")[["Quantity_Mailed", "Total_Cost"]], how="left")

summary["response_rate"] = summary["orders"] / summary["Quantity_Mailed"]
summary["CPO"] = summary["Total_Cost"] / summary["orders"]
summary["ROAS"] = summary["revenue"] / summary["Total_Cost"]



summary

,responders,orders,revenue,Quantity_Mailed,Total_Cost,response_rate,CPO,ROAS
TestCell_mail,,,,,,,,
TEST1,1392,1392,704878.64,5089,5000,0.273531,3.591954,140.975728
TEST2,467,467,224455.19,3812,2500,0.122508,5.353319,89.782076
TEST3,12,12,3417.20,12,12,1.000000,1.000000,284.766667


In [34]:
# CPO = Total Cost / Number of Orders
# ROAS = Total Revenue / Total Cost

In [35]:
fig = px.scatter(summary, x = 'CPO', y = 'ROAS', size = 'orders', color = summary.index, hover_name = summary.index, hover_data = ['orders', 'revenue', 'Total_Cost', 'response_rate'], title="CPO vs ROAS by Test Cell",
    size_max=60)
fig.show()

In [36]:
plot_df = summary.reset_index().rename(columns={"TestCell_mail": "TestCell"})
fig = px.bar(plot_df, x="TestCell", y="response_rate", title="Response Rate by Test Cell vs Holdout Baseline", labels={"response_rate": "Response Rate"})
holdout_orders = full_dataset[full_dataset["TestCell_holdout"] == "H-TEST"]["OrderNumber"].nunique()
holdout_response_rate = holdout_orders / holdout_file['FullAddress'].nunique()
fig.add_hline(y=holdout_response_rate,annotation_text="Holdout Baseline")
fig.show()

In [37]:
mailed = mail_file.groupby("TestCell")["FullAddress"].nunique()
cost = mail_plan.set_index("TestCell")["Total_Cost"]

ft = full_dataset_mail.groupby(["TestCell_mail","FirstTimeOrder"]).agg(
    responders=("CustomerID","nunique"), orders=("OrderNumber","nunique"), revenue=("OrderRevenue","sum")
)

ft["response_rate"] = ft["responders"] / ft.index.get_level_values(0).map(mailed)
ft["ROAS"] = ft["revenue"] / ft.index.get_level_values(0).map(cost)

ft

responders  orders    revenue  response_rate  \
TestCell_mail FirstTimeOrder                                                 
TEST1         0                      684     684  355105.09       0.134408   
              1                      708     708  349773.55       0.139124   
TEST2         0                      228     228  112017.30       0.059655   
              1                      239     239  112437.89       0.062533   
TEST3         0                        5       5     891.59       0.416667   
              1                        7       7    2525.61       0.583333   

                                    ROAS  
TestCell_mail FirstTimeOrder              
TEST1         0                71.021018  
              1                69.954710  
TEST2         0                44.806920  
              1                44.975156  
TEST3         0                74.299167  
              1               210.467500

5. For the test cells targeted for New Customer Acquisition, which one do you think is "best" when ignoring the holdout group's performance? Which test cell is "best" when factoring in the holdout? Is the answer still the same? Why or why not?


When ignoring the holdout group's performance, I believe Test 1 is best for new customer acquisition. This is due to its higher response rate and ROAS and lower CPO. When we factor in the holdout baseline, Test 1 still performs better in response rate. Test 2, on the otherhand did quite badly compared to Test 1 and the holdout. Test 3 is another story that we will address later. 



6. After reading these results, the client notices the "perfect" performance for Test Cell 3. They determine that since Test Cell 3 used a Foldout format, we should only mail Foldouts going forward. Do you agree?


Definitely not. We should not only mail Foldouts going forward. Test 3 suffers from a small sample size because it was only mailed to 12 people, and all 12 people ordered. Just because this tiny test size resulted in all orders being fulfilled does not mean that this will be the case in the future. Test 3 was just sent to VIP customers, so this adds a confounding variable that may show that the high response rates may be due to the customers' loyalty and less to do with the actual test. Test 2 also employed the Foldout method and performed poorly with new customers. Overall, Test 3  has high variance, making its reliability low, meaning it cannot be generalized to dictate future strategy. 



7. The client wants to plan a second mailing campaign, using these results as an aid. What are your recommendations to the client on what they should do? Be precise, in one to three sentences.

Based on the analysis done today, I would suggest the client use the postcard primarily for new customers as Test 1 demonstrated the best balance of response rate, cost efficiency, and statistical reliability at a larger sample size. In the next round of tests, I suggest running a larger, more controlled test of the follow-up format with a much broader audience to validate whether the strong performance that happened in Test 3 is driven by the format or the audience as a confounder. Additionally, I would test the postcard on the VIP customers and also make sure the holdout group is retained in order to continue to measure any incremental lift to see if results really reflect true marketing impacts. 
